# YouTube Comments Analysis — CRISP-DM
**Video:** "Can I Vibecode a $250M App Better Than a Pro Developer?" — Riley
**Comments:** ~3,007 | **Date:** 2026-04-06

In [ ]:
import json, re, warnings
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import networkx as nx
import ollama
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

DATA_RAW       = Path('data/raw/comments.json')
DATA_CLEAN     = Path('data/processed/comments_clean.csv')
EMBEDDINGS_PATH = Path('data/processed/embeddings.npy')
CLUSTERS_PATH  = Path('data/processed/clusters.csv')

CONCEPTS = [
    'ide', 'terminal', 'vibe coding', 'vibe code', 'vibecod',
    'ai', 'artificial intelligence', 'cursor', 'real developer',
    'no code', 'nocode', 'tts', 'transcription', 'coding', 'programmer',
    'fear', 'easy', 'impossible', 'future', 'replace',
    'cheat', 'shortcut', 'real coding', 'fake'
]
print("✓ Setup completo")

# Fase 1: Business Understanding

## Contexto
Riley publicó "Can I Vibecode a $250M App Better Than a Pro Developer? (With No Code)".
El video muestra a un no-programador construyendo una app real usando herramientas de IA,
sin entender conceptos como IDE, terminal, o cómo funciona la transcripción del iPhone.

## Preguntas de Negocio
1. ¿Cuál es el sentimiento general hacia el video y hacia el vibe coding?
2. ¿Qué perfil tiene la persona que comenta? (coder, vibe-coder, casual, técnico)
3. ¿Las personas técnicas sienten amenaza/escepticismo ante el vibe coding?
4. ¿Las personas no técnicas se sorprenden de que Riley no entienda cosas básicas?
5. ¿Existe brecha entre "vibe coding es el futuro" vs "no es coding real"?

## Hipótesis
- **H1:** Coders escépticos reaccionan negativamente a la ignorancia técnica de Riley
- **H2:** Vibe coders/fans entusiastas celebran que "no necesitás saber código"
- **H3:** `IDE` y `terminal` generan comentarios de incredulidad o condescendencia
- **H4:** Los técnicos no tienen miedo, sino escepticismo sobre si es "real" coding

# Fase 2: Data Understanding

In [ ]:
records = []
with open(DATA_RAW, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df_raw = pd.DataFrame(records)
print(f"Total comentarios: {len(df_raw)}")
print(f"Columnas: {list(df_raw.columns)}")
df_raw[['author', 'text', 'votes']].head(3)

In [ ]:
print("=== Info del Dataset ===")
print(f"Con likes > 0: {(df_raw['votes'] > 0).sum()} ({(df_raw['votes'] > 0).mean():.1%})")
print(f"Más likeado: {df_raw['votes'].max()} likes")
print("\nTop 5 por likes:")
df_raw.nlargest(5, 'votes')[['author', 'text', 'votes']]

In [ ]:
df_raw['char_len'] = df_raw['text'].str.len()
df_raw['word_len'] = df_raw['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_raw['char_len'].clip(upper=500), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de longitud (caracteres)')
axes[0].axvline(df_raw['char_len'].median(), color='red', linestyle='--',
                label=f'Mediana: {df_raw["char_len"].median():.0f}')
axes[0].legend()
axes[1].hist(df_raw['word_len'].clip(upper=80), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribución de longitud (palabras)')
axes[1].axvline(df_raw['word_len'].median(), color='navy', linestyle='--',
                label=f'Mediana: {df_raw["word_len"].median():.0f}')
axes[1].legend()
plt.tight_layout()
plt.savefig('data/processed/dist_longitud.png', bbox_inches='tight')
plt.show()
print(f"Mediana: {df_raw['char_len'].median():.0f} chars | {df_raw['word_len'].median():.0f} words")

In [ ]:
STOPWORDS = {
    'the','a','an','is','it','in','to','of','and','or','i','this','that','for',
    'you','he','she','they','we','with','as','at','be','was','are','have','has',
    'had','but','not','do','did','so','if','by','on','your','my','his','her',
    'its','just','can','will','would','could','like','what','when','how','who',
    'there','here','from','all','one','no','up','out','about','get','got','into',
    'more','even','than','me','him','them','us','been','were','their','our',
    'im','ive','dont','doesnt','cant','hes','shes','thats'
}
all_words = []
for text in df_raw['text']:
    words = re.findall(r'\b[a-z]{3,}\b', str(text).lower())
    all_words.extend([w for w in words if w not in STOPWORDS])

word_freq = Counter(all_words).most_common(20)
words_list, counts = zip(*word_freq)
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(list(reversed(words_list)), list(reversed(counts)), color='steelblue')
ax.set_title('Top 20 palabras más frecuentes (sin stopwords)', fontsize=14)
for bar, count in zip(bars, reversed(counts)):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2, str(count), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('data/processed/top_palabras.png', bbox_inches='tight')
plt.show()

In [ ]:
concept_coverage = {c: df_raw['text'].str.lower().str.contains(c, na=False).sum() for c in CONCEPTS}
coverage_df = (pd.DataFrame.from_dict(concept_coverage, orient='index', columns=['menciones'])
               .query('menciones > 0').sort_values('menciones', ascending=False))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(coverage_df.index, coverage_df['menciones'], color='darkorange', edgecolor='white')
ax.set_title('Menciones de conceptos técnicos clave', fontsize=14)
ax.set_ylabel('Nº comentarios')
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, coverage_df['menciones']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('data/processed/concept_coverage.png', bbox_inches='tight')
plt.show()
print(coverage_df)

# Fase 3: Data Preparation
## 3a. Limpieza

In [ ]:
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_meaningful(text: str) -> bool:
    return len(re.sub(r'[^\w]', '', text)) >= 10

df = df_raw.copy()
df['text_clean'] = df['text'].apply(clean_text)
df['meaningful'] = df['text_clean'].apply(is_meaningful)

print(f"Originales:       {len(df)}")
print(f"Sin sentido:      {(~df['meaningful']).sum()}")
df = df[df['meaningful']].drop_duplicates(subset='text_clean').reset_index(drop=True)
print(f"Después limpieza: {len(df)}")

df[['author','text','text_clean','votes','time','replies']].to_csv(DATA_CLEAN, index=False)
print(f"✓ Guardado: {DATA_CLEAN}")

## 3b. Embeddings

In [ ]:
print("Cargando sentence-transformers...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Dim: {embed_model.get_sentence_embedding_dimension()}")

print(f"\nGenerando embeddings para {len(df)} comentarios (1-3 min CPU)...")
embeddings = embed_model.encode(
    df['text_clean'].tolist(),
    batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
np.save(EMBEDDINGS_PATH, embeddings)
print(f"✓ Shape: {embeddings.shape} | Guardado: {EMBEDDINGS_PATH}")

# Fase 4: Modeling
## 4a. Sentiment Analysis (VADER)

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text: str) -> dict:
    s = sia.polarity_scores(text)
    c = s['compound']
    return {'compound': c, 'sentiment': 'positive' if c>=0.05 else ('negative' if c<=-0.05 else 'neutral'),
            'pos': s['pos'], 'neg': s['neg'], 'neu': s['neu']}

df = pd.concat([df, df['text_clean'].apply(get_sentiment).apply(pd.Series)], axis=1)
print(df['sentiment'].value_counts())
print(f"\nCompound promedio: {df['compound'].mean():.3f}")

In [ ]:
sent_counts = df['sentiment'].value_counts()
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(sent_counts.values, labels=sent_counts.index,
            colors=[colors[s] for s in sent_counts.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize':12})
axes[0].set_title('Sentimiento General', fontsize=14)

axes[1].hist(df['compound'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
for v, c, lbl in [(0.05,'green','Umbral pos'), (-0.05,'red','Umbral neg')]:
    axes[1].axvline(v, color=c, linestyle='--', alpha=0.7, label=lbl)
axes[1].axvline(df['compound'].mean(), color='orange', linewidth=2,
                label=f'Media: {df["compound"].mean():.3f}')
axes[1].set_title('Compound Score (VADER)'); axes[1].legend()
plt.tight_layout()
plt.savefig('data/processed/sentiment_general.png', bbox_inches='tight')
plt.show()

In [ ]:
aspect_results = []
for concept in CONCEPTS:
    mask = df['text_clean'].str.lower().str.contains(concept, na=False)
    sub = df[mask]
    if len(sub) >= 5:
        aspect_results.append({'concept': concept, 'count': len(sub),
            'mean_compound': sub['compound'].mean(),
            'pct_positive': (sub['sentiment']=='positive').mean(),
            'pct_negative': (sub['sentiment']=='negative').mean()})

aspect_df = pd.DataFrame(aspect_results).sort_values('mean_compound')
fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in aspect_df['mean_compound']]
bars = ax.barh(aspect_df['concept'], aspect_df['mean_compound'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Sentimiento por Concepto Técnico (Aspect-Based)', fontsize=14)
for bar, count in zip(bars, aspect_df['count']):
    x = bar.get_width()
    ax.text(x+(0.005 if x>=0 else -0.005), bar.get_y()+bar.get_height()/2,
            f'n={count}', va='center', fontsize=9, ha='left' if x>=0 else 'right')
plt.tight_layout()
plt.savefig('data/processed/aspect_sentiment.png', bbox_inches='tight')
plt.show()
print(aspect_df.to_string(index=False))

## 4b. Clustering: UMAP + HDBSCAN

In [ ]:
print("UMAP 10D (clustering)...")
reducer_10d = umap.UMAP(n_components=10, n_neighbors=20, min_dist=0.0, metric='cosine', random_state=42)
embedding_10d = reducer_10d.fit_transform(embeddings)
print(f"✓ {embedding_10d.shape}")

In [ ]:
print("UMAP 2D (visualización)...")
reducer_2d = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1, metric='cosine', random_state=42)
embedding_2d = reducer_2d.fit_transform(embeddings)
print(f"✓ {embedding_2d.shape}")

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10,
                             metric='euclidean', cluster_selection_method='eom')
cluster_labels = clusterer.fit_predict(embedding_10d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
print(f"✓ Clusters: {n_clusters}")
for label in sorted(set(cluster_labels)):
    name = "RUIDO" if label == -1 else f"Cluster {label}"
    count = (cluster_labels == label).sum()
    print(f"  {name}: {count} ({count/len(cluster_labels):.1%})")

df['cluster'] = cluster_labels
df['umap_x'] = embedding_2d[:, 0]
df['umap_y'] = embedding_2d[:, 1]
df.to_csv(CLUSTERS_PATH, index=False)
print(f"✓ Guardado: {CLUSTERS_PATH}")

In [ ]:
df['cluster_str'] = df['cluster'].apply(lambda x: 'Ruido' if x==-1 else f'Cluster {x}')
fig = px.scatter(df, x='umap_x', y='umap_y', color='cluster_str',
    hover_data={'text_clean':True,'sentiment':True,'votes':True,'umap_x':False,'umap_y':False},
    title='Clusters de Comentarios (UMAP 2D + HDBSCAN)', opacity=0.6, width=900, height=600)
fig.update_traces(marker=dict(size=4))
fig.show()

In [ ]:
def get_representative_comments(cluster_id: int, n: int = 10) -> pd.DataFrame:
    mask = df['cluster'] == cluster_id
    cluster_embs = embeddings[mask]
    centroid = cluster_embs.mean(axis=0, keepdims=True)
    sims = cosine_similarity(centroid, cluster_embs)[0]
    top_idx = sims.argsort()[-n:][::-1]
    return df[mask].iloc[top_idx][['text_clean','votes','sentiment','compound']]

for label in sorted([l for l in set(cluster_labels) if l != -1]):
    print(f"\n{'='*60}\nCLUSTER {label} — Top 10\n{'='*60}")
    for _, row in get_representative_comments(label).iterrows():
        print(f"[{row['sentiment']:8s}|{row['compound']:+.2f}|👍{row['votes']}] {row['text_clean'][:120]}")

In [ ]:
def label_cluster_with_llm(cluster_id: int) -> str:
    reps = get_representative_comments(cluster_id, n=15)
    sample = "\n".join([f"- {t[:150]}" for t in reps['text_clean']])
    prompt = (f"YouTube comments about vibe coding (non-programmer builds app with AI, unaware of IDE/terminal).\n"
              f"Cluster of {len(df[df['cluster']==cluster_id])} similar comments:\n{sample}\n\n"
              "1. What type of person writes these?\n"
              "2. Their attitude toward vibe coding?\n"
              "3. Short persona label (e.g. 'Skeptical Developer'). Max 80 words.")
    r = ollama.chat(model='qwen3:1.7b', messages=[{'role':'user','content':prompt}],
                    options={'temperature':0.3})
    return r['message']['content']

cluster_labels_dict = {}
for label in sorted([l for l in set(cluster_labels) if l != -1]):
    print(f"\n--- Cluster {label} ---")
    desc = label_cluster_with_llm(label)
    cluster_labels_dict[label] = desc
    print(desc)

## 4c. Grafo de Co-ocurrencia de Conceptos

In [ ]:
df['concepts_found'] = df['text_clean'].apply(
    lambda t: [c for c in CONCEPTS if c in t.lower()])

concept_freq = Counter()
co_occurrence = Counter()
for concepts in df['concepts_found']:
    concept_freq.update(concepts)
    if len(concepts) >= 2:
        for pair in combinations(sorted(concepts), 2):
            co_occurrence[pair] += 1

print(f"Nodos: {sum(1 for n in concept_freq.values() if n > 0)}, Aristas: {len(co_occurrence)}")
print("\nTop 10 conceptos:")
for c, n in concept_freq.most_common(10): print(f"  {c}: {n}")
print("\nTop 10 co-ocurrencias:")
for (c1,c2), n in co_occurrence.most_common(10): print(f"  {c1} ↔ {c2}: {n}")

In [ ]:
G = nx.Graph()
for concept, freq in concept_freq.items():
    if freq >= 3: G.add_node(concept, frequency=freq)
for (c1, c2), weight in co_occurrence.items():
    if weight >= 2 and G.has_node(c1) and G.has_node(c2):
        G.add_edge(c1, c2, weight=weight)

print(f"Grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
pos = nx.spring_layout(G, k=2.5, seed=42, weight='weight')
node_sizes = [concept_freq[n] * 8 for n in G.nodes()]
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1

fig, ax = plt.subplots(figsize=(14, 10))
centrality = nx.degree_centrality(G)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                       node_color=[centrality[n] for n in G.nodes()],
                       cmap=plt.cm.YlOrRd, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, width=[w/max_w*6 for w in edge_weights],
                       alpha=0.4, edge_color='gray', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
ax.set_title('Grafo de Co-ocurrencia de Conceptos', fontsize=14); ax.axis('off')
sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=plt.Normalize(0,1))
plt.colorbar(sm, ax=ax, label='Centralidad de Grado', shrink=0.5)
plt.tight_layout()
plt.savefig('data/processed/concept_graph.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
degree_c = nx.degree_centrality(G)
between_c = nx.betweenness_centrality(G, weight='weight')
eigen_c   = nx.eigenvector_centrality(G, weight='weight', max_iter=500)

centrality_df = pd.DataFrame({
    'concept': list(G.nodes()),
    'frequency': [concept_freq[n] for n in G.nodes()],
    'degree': [degree_c[n] for n in G.nodes()],
    'betweenness': [between_c[n] for n in G.nodes()],
    'eigenvector': [eigen_c[n] for n in G.nodes()],
}).sort_values('degree', ascending=False)
print("Top 10 conceptos más conectados:")
print(centrality_df.head(10).to_string(index=False))

# Fase 5: Evaluation

In [ ]:
non_noise = df['cluster'] != -1
if non_noise.sum() > 1:
    sil = silhouette_score(embeddings[non_noise], df.loc[non_noise,'cluster'],
                           metric='cosine', sample_size=1000, random_state=42)
    print(f"Silhouette Score: {sil:.4f}  (>0.3 aceptable para texto)")

summary = df[non_noise].groupby('cluster').agg(
    tamaño=('text_clean','count'),
    compound_mean=('compound','mean'),
    pct_pos=('sentiment', lambda x: (x=='positive').mean()),
    pct_neg=('sentiment', lambda x: (x=='negative').mean()),
    likes_mean=('votes','mean'),
    likes_max=('votes','max'),
).round(3)
print("\nResumen por Cluster:")
print(summary.to_string())

In [ ]:
cross = pd.crosstab(df[df['cluster']!=-1]['cluster'],
                    df[df['cluster']!=-1]['sentiment'], normalize='index') * 100
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(cross, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label':'% comentarios'})
ax.set_title('Sentimiento por Cluster (%)', fontsize=13)
plt.tight_layout()
plt.savefig('data/processed/cluster_sentiment_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("VALIDACIÓN DE HIPÓTESIS")
print("=" * 60)
for concept in ['ide', 'terminal', 'vibe coding', 'real developer', 'replace', 'fear']:
    mask = df['text_clean'].str.lower().str.contains(concept, na=False)
    if mask.sum() >= 5:
        avg = df[mask]['compound'].mean()
        pct_neg = (df[mask]['sentiment']=='negative').mean() * 100
        print(f"\n'{concept}' ({mask.sum()} comentarios):")
        print(f"  Compound promedio: {avg:+.3f} | % negativos: {pct_neg:.1f}%")
print("\n>> Completar conclusiones H1-H4 con los números reales obtenidos <<")

# Fase 6: Deployment — Mini RAG con qwen3:1.7b

In [ ]:
def ask(query: str, k: int = 15, model: str = 'qwen3:1.7b') -> str:
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ query_emb.T).flatten()
    top_idx = scores.argsort()[-k:][::-1]
    context = "\n".join(
        f"[{row['sentiment']}|cluster={row['cluster']}] {row['text_clean']}"
        for _, row in df.iloc[top_idx][['text_clean','sentiment','cluster']].iterrows()
    )
    prompt = (f"You analyze YouTube comments about vibe coding "
              f"(non-programmer builds app with AI, unaware of IDE/terminal).\n\n"
              f"{k} relevant comments:\n{context}\n\n"
              f"Question: {query}\n"
              f"Answer analytically. Cite 2-3 specific comments.")
    r = ollama.chat(model=model, messages=[{'role':'user','content':prompt}],
                    options={'temperature':0.3})
    return r['message']['content']

try:
    test = ollama.chat(model='qwen3:1.7b', messages=[{'role':'user','content':'Say OK'}],
                       options={'temperature':0})
    print(f"✓ Ollama OK: {test['message']['content']}")
except Exception as e:
    print(f"✗ Error: {e} — ejecutá 'ollama serve' en otra terminal")

In [ ]:
queries = [
    "What do professional developers think about vibe coding?",
    "Are people surprised that Riley doesn't know what an IDE or terminal is?",
    "Do people think AI tools will replace real programming skills?",
    "What do non-technical people say about the video?",
    "Is there a debate about whether vibe coding is real coding?"
]
for q in queries:
    print(f"\n{'='*60}\nQ: {q}\n{'='*60}")
    print(ask(q))

In [ ]:
# Modo interactivo — escribí tus propias preguntas
while True:
    query = input("\n🔍 Tu pregunta (o 'exit'): ").strip()
    if query.lower() in ('exit', 'quit', 'q', ''):
        break
    print("\n" + ask(query))